In [13]:
# ── Cell 1: Install Google Chrome + libraries ─────────────────
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q
!pip install selenium webdriver-manager beautifulsoup4 -q

Reading package lists...
Building dependency tree...
Reading state information...
google-chrome-stable is already the newest version (149.0.7827.53-1).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.


In [15]:
# ── Cell 2: Full scraper ──────────────────────────────────────
import time
import csv
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup


def init_driver():
    """Initialize a headless Chrome WebDriver configured for Colab."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
    options.binary_location = "/usr/bin/google-chrome"

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver


def load_page(driver, url):
    """Load the page and wait for product cards to render."""
    driver.get(url)
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".thumbnail"))
        )
    except Exception:
        print("Timed out waiting for product elements; continuing anyway.")

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    return driver.page_source


def parse_products(html):
    """Parse product details from the rendered HTML."""
    soup = BeautifulSoup(html, "html.parser")
    products = []
    cards = soup.select(".thumbnail")  # each product card

    for card in cards:
        name_el = card.select_one("a.title")
        price_el = card.select_one(".price")
        desc_el = card.select_one(".description")

        # Use the link's title attribute for the full, untruncated name
        if name_el:
            name = name_el.get("title", name_el.get_text(strip=True))
        else:
            name = "N/A"

        products.append({
            "name": name,
            "price": price_el.get_text(strip=True) if price_el else "N/A",
            "features": desc_el.get_text(strip=True) if desc_el else "N/A",
        })
    return products


def save_data(products, filename="products.csv"):
    """Save scraped products to a CSV file."""
    if not products:
        print("No data to save.")
        return
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "price", "features"])
        writer.writeheader()
        writer.writerows(products)
    print(f"Saved {len(products)} products to {filename}")


def main():
    url = "https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops"
    driver = init_driver()
    try:
        html = load_page(driver, url)
        products = parse_products(html)
        for p in products:
            print(p)
        save_data(products)
    finally:
        driver.quit()


main()

{'name': 'Asus VivoBook X441NA-GA190', 'price': '$295.99', 'features': 'Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd'}
{'name': 'Prestigio SmartBook 133S Dark Grey', 'price': '$299', 'features': 'Prestigio SmartBook 133S Dark Grey, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam'}
{'name': 'Prestigio SmartBook 133S Gold', 'price': '$299', 'features': 'Prestigio SmartBook 133S Gold, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam'}
{'name': 'Aspire E1-510', 'price': '$306.99', 'features': '15.6", Pentium N3520 2.16GHz, 4GB, 500GB, Linux'}
{'name': 'Lenovo V110-15IAP', 'price': '$321.94', 'features': 'Lenovo V110-15IAP, 15.6" HD, Celeron N3350 1.1GHz, 4GB, 128GB SSD, Windows 10 Home'}
{'name': 'Lenovo V110-15IAP', 'price': '$356.49', 'features': 'Asus VivoBook 15 X540NA-GQ008T Chocolate Black, 15.6" HD, Pentium N4200, 4GB, 500GB, Windows 10 Home, En kbd'}
{'name'

In [16]:
# ── Cell 3: Download the CSV ──────────────────────────────────
from google.colab import files
files.download("products.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>